## Search Engine With Tools and Agents

In [ ]:
## Arxiv -- Research

## Tools creation

from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [ ]:
api_wrapper_wikki = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=1000)

wiki_tool=WikipediaQueryRun(api_wrapper=api_wrapper_wikki)

wiki_tool.name

In [ ]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=1000)
arxiv_tool=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

arxiv_tool.name

In [ ]:
tools = [wiki_tool,arxiv_tool]

In [ ]:
## Custom tools [RAG Tool]

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
groq_api_key = os.getenv("GROQ_API_KEY")
## Langsmith Tracking
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_API_KEY"]

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings =HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
loader = WebBaseLoader("https://developers.openai.com/api/docs/guides/prompt-engineering")
data = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(data)

vectordb = Chroma.from_documents(documents=documents,embedding=embeddings,persist_directory="./chroma_db")

retriever = vectordb.as_retriever()

retriever

In [ ]:
from langchain_core.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(retriever=retriever, name="RAGTool", description="Useful for retrieving information from the web")

retriever_tool.name

In [ ]:
tools = [wiki_tool,arxiv_tool,retriever_tool]

In [ ]:
# Run all this tools with agents and LLMs 

## Tools, LLM -> AgentExecutor

llm = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=groq_api_key)

In [ ]:
from langchain_classic import hub

prompt =hub.pull("hwchase17/openai-functions-agent")

prompt.messages

In [ ]:
# Agents

from langchain_classic.agents import create_openai_tools_agent

agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)

agent

In [ ]:
## Agent executor
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

In [ ]:
agent_executor.invoke({"input": "Explain Open AI Prompt engineering"})

In [ ]:
agent_executor.invoke({"input": "who is pm of france?"})

In [ ]:
agent_executor.invoke({"input": "what are the latest research papers on transformers?"})